# Figure 3-style coincidence analysis: biological neuron vs NEST models

This notebook implements a held-out predictive comparison inspired by Figure 3 of Kobayashi et al. (2009).

The workflow is:

1. Load the injected current and repeated biological voltage recordings.
2. Keep the biological repetitions separate.
3. Use the first 25 s for fitting and the remaining 14 s for validation.
4. Fit a small number of model-specific parameters by maximising the mean coincidence factor on fitting windows.
5. Simulate every fitted model on the untouched validation current.
6. Compute:
   - raw model–experiment coincidence, \(\Gamma\);
   - biological trial-to-trial reliability;
   - normalised predictive score, \(\Gamma_A\).

The 21 s challenge current without biological voltage is retained only for later blind prediction. It cannot be scored locally.

## Cell 1 — Imports and reproducibility

The optimisation uses SciPy's differential evolution because the spike-timing objective is discontinuous and therefore unsuitable for ordinary gradient descent.

NEST is reset before every simulation so that membrane and threshold states never leak between candidates or data windows.

In [ ]:
from pathlib import Path
from itertools import combinations
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nest
import pynestml
from scipy.optimize import differential_evolution
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

SEED = 12345
rng = np.random.default_rng(SEED)

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 100)

print("NEST version:", nest.__version__)
print("NumPy version:", np.__version__)

## Cell 2 — Spike extraction and coincidence helpers

A biological spike is detected at the upward crossing of a voltage threshold.

Coincidences are matched one-to-one within \(\pm 2\) ms. The chance correction uses the firing rate of the comparison/model spike train, as in the coincidence-factor definition.

In [ ]:
def extract_biological_spikes(voltage_trace, dt_ms=0.1, threshold_mv=-10.0):
    """Return upward threshold-crossing times in milliseconds."""
    v = np.asarray(voltage_trace, dtype=float).reshape(-1)

    if len(v) < 2:
        return np.array([], dtype=float)

    crossings = np.flatnonzero(
        (v[:-1] < threshold_mv) &
        (v[1:] >= threshold_mv)
    )

    return (crossings + 1) * dt_ms


def count_coincident_spikes(data_spikes, model_spikes, delta_ms=2.0):
    """Count one-to-one spike coincidences within ±delta_ms."""
    data_spikes = np.sort(np.asarray(data_spikes, dtype=float))
    model_spikes = np.sort(np.asarray(model_spikes, dtype=float))

    i = 0
    j = 0
    n_coinc = 0

    while i < len(data_spikes) and j < len(model_spikes):
        difference = model_spikes[j] - data_spikes[i]

        if abs(difference) <= delta_ms:
            n_coinc += 1
            i += 1
            j += 1
        elif difference < -delta_ms:
            j += 1
        else:
            i += 1

    return n_coinc


def calculate_coincidence_factor(
    model_spikes,
    data_spikes,
    duration_ms,
    delta_ms=2.0,
):
    """Calculate the chance-corrected coincidence factor Gamma."""
    model_spikes = np.asarray(model_spikes, dtype=float)
    data_spikes = np.asarray(data_spikes, dtype=float)

    n_model = len(model_spikes)
    n_data = len(data_spikes)

    if duration_ms <= 0:
        raise ValueError("duration_ms must be positive.")

    if n_model + n_data == 0:
        return np.nan

    duration_sec = duration_ms / 1000.0
    delta_sec = delta_ms / 1000.0

    # Rate of the model/comparison train.
    nu = n_model / duration_sec

    expected_coincidences = 2.0 * nu * delta_sec * n_data
    chance_normalisation = 1.0 - 2.0 * nu * delta_sec

    if chance_normalisation <= 0:
        return np.nan

    n_coinc = count_coincident_spikes(
        data_spikes=data_spikes,
        model_spikes=model_spikes,
        delta_ms=delta_ms,
    )

    gamma = (
        (n_coinc - expected_coincidences)
        / (0.5 * (n_data + n_model))
        / chance_normalisation
    )

    return float(gamma)


def calculate_experimental_reliability(
    voltage_repetitions,
    dt_ms=0.1,
    threshold_mv=-10.0,
    delta_ms=2.0,
):
    """
    Mean pairwise experiment–experiment Gamma over repeated trials.

    Both directional scores are averaged because the chance correction
    depends on which train is treated as the comparison train.
    """
    voltage_repetitions = np.asarray(voltage_repetitions, dtype=float)

    if voltage_repetitions.ndim != 2:
        raise ValueError("Expected samples × repetitions.")

    duration_ms = voltage_repetitions.shape[0] * dt_ms

    spike_trains = [
        extract_biological_spikes(
            voltage_repetitions[:, repetition],
            dt_ms=dt_ms,
            threshold_mv=threshold_mv,
        )
        for repetition in range(voltage_repetitions.shape[1])
    ]

    pairwise_scores = []

    for first, second in combinations(range(len(spike_trains)), 2):
        forward = calculate_coincidence_factor(
            model_spikes=spike_trains[second],
            data_spikes=spike_trains[first],
            duration_ms=duration_ms,
            delta_ms=delta_ms,
        )

        reverse = calculate_coincidence_factor(
            model_spikes=spike_trains[first],
            data_spikes=spike_trains[second],
            duration_ms=duration_ms,
            delta_ms=delta_ms,
        )

        pairwise_scores.append(np.nanmean([forward, reverse]))

    return {
        "reliability": float(np.nanmean(pairwise_scores)),
        "pairwise_scores": np.asarray(pairwise_scores),
        "spike_trains": spike_trains,
    }

## Cell 3 — Load the challenge data

The current file contains 60 s of input. The voltage file contains only the first 39 s, with each column representing a repeated biological response to the same injected current.

The voltage repetitions are not averaged.

In [ ]:
# Load data
CURRENT_PATH = r"data_challengeA/current.csv"
VOLTAGE_PATH = r"data_challengeA/voltage_allrep.csv"

raw_current = pd.read_csv(CURRENT_PATH,header=None,names=["Current_pA"])
raw_voltage = pd.read_csv(VOLTAGE_PATH, header=None, sep=r"\s+")

current_pA = raw_current["Current_pA"].to_numpy(dtype=float)
voltage_mv = raw_voltage.to_numpy(dtype=float)

DT_MS = 0.1
DT_S = DT_MS / 1000.0

current_time_s = np.arange(len(current_pA)) * DT_S
voltage_time_s = np.arange(len(voltage_mv)) * DT_S

print(f"Current shape: {current_pA.shape}")
print(f"Voltage shape: {voltage_mv.shape}")
print(f"Biological repetitions: {voltage_mv.shape[1]}")
print(f"Current duration: {len(current_pA) * DT_S:.1f} s")
print(f"Voltage duration: {len(voltage_mv) * DT_S:.1f} s")

if len(voltage_mv) > len(current_pA):
    raise ValueError("Voltage recording is longer than the supplied current.")

## Cell 4 — Inspect the raw current and repeated voltage traces

Only a short voltage interval is plotted so that individual spikes remain visible. This also provides a quick check that the time step and voltage units are sensible.

In [ ]:
preview_start_s = 0.0
preview_end_s = 2.0

preview_slice = slice(
    int(preview_start_s / DT_S),
    int(preview_end_s / DT_S),
)

plt.figure(figsize=(12, 3.5))
plt.plot(
    current_time_s[preview_slice],
    current_pA[preview_slice],
)
plt.xlabel("Time (s)")
plt.ylabel("Injected current (pA)")
plt.title("Injected current: first 2 seconds")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
for repetition in range(min(5, voltage_mv.shape[1])):
    plt.plot(
        voltage_time_s[preview_slice],
        voltage_mv[preview_slice, repetition],
        linewidth=0.8,
        label=f"Repetition {repetition + 1}",
    )

plt.axhline(-10.0, linestyle="--", linewidth=1, label="Spike threshold")
plt.xlabel("Time (s)")
plt.ylabel("Membrane voltage (mV)")
plt.title("Repeated biological responses: first 2 seconds")
plt.legend(ncol=3)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Cell 5 — Create fitting, validation and blind-test datasets

The 39 s with biological voltage are split contiguously:

- **Fitting:** 0–25 s
- **Validation:** 25–39 s
- **Blind challenge current:** 39–60 s

A contiguous validation interval prevents information leakage. The 21 s blind current has no local voltage target and is therefore not used to calculate a score.

In [ ]:
FIT_START_S = 0.0
FIT_END_S = 25.0

VALIDATION_START_S = 25.0
VALIDATION_END_S = 39.0

BLIND_START_S = 39.0
BLIND_END_S = min(60.0, len(current_pA) * DT_S)


def seconds_to_slice(start_s, end_s, dt_s=DT_S):
    start_index = int(round(start_s / dt_s))
    end_index = int(round(end_s / dt_s))
    return slice(start_index, end_index)


fit_slice = seconds_to_slice(FIT_START_S, FIT_END_S)
validation_slice = seconds_to_slice(
    VALIDATION_START_S,
    VALIDATION_END_S,
)
blind_slice = seconds_to_slice(BLIND_START_S, BLIND_END_S)

I_fit = current_pA[fit_slice]
V_fit = voltage_mv[fit_slice, :]

I_validation = current_pA[validation_slice]
V_validation = voltage_mv[validation_slice, :]

I_blind = current_pA[blind_slice]

print("Fitting")
print("-------")
print(f"Current samples: {len(I_fit):,}")
print(f"Voltage shape: {V_fit.shape}")
print(f"Duration: {len(I_fit) * DT_S:.1f} s")

print("\nValidation")
print("----------")
print(f"Current samples: {len(I_validation):,}")
print(f"Voltage shape: {V_validation.shape}")
print(f"Duration: {len(I_validation) * DT_S:.1f} s")

print("\nBlind challenge current")
print("-----------------------")
print(f"Current samples: {len(I_blind):,}")
print(f"Duration: {len(I_blind) * DT_S:.1f} s")

assert len(I_fit) == len(V_fit)
assert len(I_validation) == len(V_validation)

## Cell 6 — Detect biological spikes and check the detector

The detector is applied separately to every repetition. The table reports firing rates, and the overlay checks whether the selected threshold captures one event per action potential.

Adjust `BIOLOGICAL_THRESHOLD_MV` only if inspection shows missed spikes or false crossings.

In [ ]:
BIOLOGICAL_THRESHOLD_MV = -10.0
DELTA_MS = 2.0

fit_biological_spikes = [
    extract_biological_spikes(
        V_fit[:, repetition],
        dt_ms=DT_MS,
        threshold_mv=BIOLOGICAL_THRESHOLD_MV,
    )
    for repetition in range(V_fit.shape[1])
]

validation_biological_spikes = [
    extract_biological_spikes(
        V_validation[:, repetition],
        dt_ms=DT_MS,
        threshold_mv=BIOLOGICAL_THRESHOLD_MV,
    )
    for repetition in range(V_validation.shape[1])
]

fit_duration_s = len(I_fit) * DT_S
validation_duration_s = len(I_validation) * DT_S

spike_summary = pd.DataFrame({
    "repetition": np.arange(1, V_fit.shape[1] + 1),
    "fit_spikes": [len(s) for s in fit_biological_spikes],
    "fit_rate_hz": [len(s) / fit_duration_s for s in fit_biological_spikes],
    "validation_spikes": [len(s) for s in validation_biological_spikes],
    "validation_rate_hz": [
        len(s) / validation_duration_s
        for s in validation_biological_spikes
    ],
})

display(spike_summary.round(3))

check_start_s = 25.0
check_end_s = 26.0
check_slice = seconds_to_slice(check_start_s, check_end_s)

check_voltage = voltage_mv[check_slice, 0]
check_time_s = voltage_time_s[check_slice]
check_spikes_ms = extract_biological_spikes(
    check_voltage,
    dt_ms=DT_MS,
    threshold_mv=BIOLOGICAL_THRESHOLD_MV,
)

plt.figure(figsize=(12, 4))
plt.plot(check_time_s, check_voltage, linewidth=0.8)
plt.axhline(
    BIOLOGICAL_THRESHOLD_MV,
    linestyle="--",
    linewidth=1,
    label="Detection threshold",
)

for spike_ms in check_spikes_ms:
    plt.axvline(
        check_start_s + spike_ms / 1000.0,
        linewidth=0.6,
        alpha=0.5,
    )

plt.xlabel("Time (s)")
plt.ylabel("Membrane voltage (mV)")
plt.title("Spike-detection check: validation repetition 1")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Cell 7 — Calculate biological reliability on the held-out interval

The denominator of \(\Gamma_A\) is calculated only from the validation interval. This ensures that model performance and biological repeatability refer to exactly the same current and duration.

In [ ]:
validation_reliability = calculate_experimental_reliability(
    V_validation,
    dt_ms=DT_MS,
    threshold_mv=BIOLOGICAL_THRESHOLD_MV,
    delta_ms=DELTA_MS,
)

R_VALIDATION = validation_reliability["reliability"]

print(f"Validation experiment–experiment reliability: {R_VALIDATION:.4f}")
print(
    "Pairwise reliability range: "
    f"{np.nanmin(validation_reliability['pairwise_scores']):.4f} to "
    f"{np.nanmax(validation_reliability['pairwise_scores']):.4f}"
)

if not np.isfinite(R_VALIDATION) or R_VALIDATION <= 0:
    raise RuntimeError(
        "Validation reliability is non-positive. "
        "Inspect spike detection and the dataset before normalising scores."
    )

## Cell 8 — Compile and install the custom NESTML AMAT model

Compilation occurs once. The module is reinstalled after each NEST kernel reset whenever the custom AMAT model is simulated.

Update `AMAT_NESTML_PATH` if your model file is stored elsewhere.

In [ ]:
AMAT_NESTML_PATH = Path("../neurons_nestml/amat_neuron.nestml")

if not AMAT_NESTML_PATH.exists():
    raise FileNotFoundError(
        f"NESTML model not found: {AMAT_NESTML_PATH.resolve()}"
    )

compiled_mod_name, custom_neuron_name = (
    NESTCodeGeneratorUtils.generate_code_for(
        str(AMAT_NESTML_PATH),
        module_name="nestml_amat_module",
        logging_level="ERROR",
    )
)

print("Compiled module:", compiled_mod_name)
print("Custom neuron:", custom_neuron_name)

## Cell 9 — NEST simulation helper

This function:

1. resets the NEST kernel;
2. reinstalls the custom module when required;
3. filters the requested parameter dictionary to parameters supported by the selected model;
4. injects the supplied current waveform;
5. returns model spike times in milliseconds.

Unsupported parameters are reported rather than silently causing the whole optimisation to fail.

In [ ]:
CUSTOM_MODELS = {custom_neuron_name}


def initialise_nest(model_name):
    nest.ResetKernel()
    nest.set_verbosity("M_ERROR")
    nest.SetKernelStatus({
        "resolution": DT_MS,
        "local_num_threads": 1,
        "rng_seed": SEED,
    })

    if model_name in CUSTOM_MODELS:
        nest.Install(compiled_mod_name)


def supported_parameter_subset(model_name, parameters):
    defaults = nest.GetDefaults(model_name)
    supported = {
        name: value
        for name, value in parameters.items()
        if name in defaults
    }

    ignored = sorted(set(parameters) - set(supported))
    return supported, ignored


def simulate_nest_model(
    model_name,
    current_values_pA,
    parameters,
    report_ignored=False,
):
    """Simulate one NEST neuron and return its spike times in ms."""
    current_values_pA = np.asarray(current_values_pA, dtype=float)

    if current_values_pA.ndim != 1:
        raise ValueError("Current must be a one-dimensional array.")

    initialise_nest(model_name)

    supported_parameters, ignored = supported_parameter_subset(
        model_name,
        parameters,
    )

    if report_ignored and ignored:
        print(f"{model_name}: ignored unsupported parameters: {ignored}")

    neuron = nest.Create(
        model_name,
        params=supported_parameters,
    )

    spike_recorder = nest.Create("spike_recorder")

    # NEST requires strictly positive, strictly increasing change times.
    amplitude_times = np.arange(
        1,
        len(current_values_pA) + 1,
        dtype=float,
    ) * DT_MS

    current_generator = nest.Create(
        "step_current_generator",
        params={
            "amplitude_times": amplitude_times,
            "amplitude_values": current_values_pA,
        },
    )

    nest.Connect(current_generator, neuron)
    nest.Connect(neuron, spike_recorder)

    duration_ms = len(current_values_pA) * DT_MS
    nest.Simulate(duration_ms)

    events = spike_recorder.get("events")
    return np.asarray(events.get("times", []), dtype=float)

# Define model structures and fitted parameters

The MAT structural values are taken from the paper where the NEST implementation supports them:

- $\tau_m=5\$ ms
- $\tau_1=10\$ ms
- $\tau_2=200\$ ms
- refractory period $\=2\$ ms

Only neuron-specific threshold parameters are fitted for MAT. AMAT fits the same parameters plus the voltage dependency term beta 
The exact NEST and NESTML parameter names can differ. The inspection table below shows which requested values each installed model actually accepts.

In [1]:
MODEL_SPECS = {
    "MAT": {
        "nest_model": "mat2_psc_exp",
        "base_params": {
            "tau_m": 5.0,
            "tau_1": 10.0,
            "tau_2": 200.0,
            "t_ref": 2.0,
            "E_L": 0.0,
            "V_m": 0.0,
        },
        "fit_names": ["alpha_1", "alpha_2", "omega"],
        "bounds": [
            (0.0, 80.0),   # alpha_1, mV
            (0.0, 10.0),   # alpha_2, mV
            (0.0, 40.0),   # omega, mV relative to baseline
        ],
    },

    "AMAT": {
        "nest_model": custom_neuron_name,
        "base_params": {
            "tau_m": 5.0,
            "tau_1": 10.0,
            "tau_2": 200.0,
            "t_ref": 2.0,
            "E_L": 0.0,
            "V_m": 0.0,
        },
        "fit_names": ["alpha_1", "alpha_2", "omega", "beta"],
        "bounds": [
            (0.0, 80.0),
            (0.0, 10.0),
            (0.0, 40.0),
            (-1.0, 1.0),
        ],
    },

    "LIF": {
        "nest_model": "iaf_psc_exp",
        "base_params": {
            "tau_m": 5.0,
            "t_ref": 2.0,
            "E_L": 0.0,
            "V_reset": 0.0,
            "V_m": 0.0,
        },
        "fit_names": ["V_th"],
        "bounds": [
            (1.0, 50.0),
        ],
    },

    "Izhikevich": {
        "nest_model": "izhikevich",
        "base_params": {},
        "fit_names": ["a", "b", "c", "d"],
        "bounds": [
            (0.005, 0.2),
            (0.05, 0.35),
            (-80.0, -40.0),
            (0.0, 20.0),
        ],
    },
}


inspection_rows = []

for label, spec in MODEL_SPECS.items():
    initialise_nest(spec["nest_model"])
    defaults = nest.GetDefaults(spec["nest_model"])

    requested = {
        **spec["base_params"],
        **dict(zip(spec["fit_names"], [np.nan] * len(spec["fit_names"]))),
    }

    for parameter_name in requested:
        inspection_rows.append({
            "label": label,
            "nest_model": spec["nest_model"],
            "parameter": parameter_name,
            "supported": parameter_name in defaults,
            "default_value": defaults.get(parameter_name, np.nan),
        })

parameter_inspection = pd.DataFrame(inspection_rows)
display(parameter_inspection)

unsupported_fitted = parameter_inspection[
    (~parameter_inspection["supported"]) &
    (parameter_inspection.apply(
        lambda row: row["parameter"] in MODEL_SPECS[row["label"]]["fit_names"],
        axis=1,
    ))
]

if len(unsupported_fitted):
    raise KeyError(
        "At least one fitted parameter is unsupported. "
        "Edit MODEL_SPECS to match the installed model names:\n"
        + unsupported_fitted.to_string(index=False)
    )

NameError: name 'custom_neuron_name' is not defined

## Cell 11 — Choose practical fitting windows

Optimising every candidate on the full 25 s trace is computationally expensive. Instead, several separated, contiguous fitting windows are used.

Each window is simulated independently from a clean state. They are never concatenated, so the model does not inherit an artificial membrane or threshold state between distant portions of the recording.

Increase the number or duration of windows for the final analysis after the pipeline works.

In [ ]:
FIT_WINDOW_SECONDS = [
    (1.0, 4.0),
    (10.0, 13.0),
    (20.0, 23.0),
]


def relative_seconds_to_slice(start_s, end_s):
    return slice(
        int(round(start_s / DT_S)),
        int(round(end_s / DT_S)),
    )


fitting_windows = []

for start_s, end_s in FIT_WINDOW_SECONDS:
    window_slice = relative_seconds_to_slice(start_s, end_s)

    window_current = I_fit[window_slice]
    window_voltage = V_fit[window_slice, :]

    window_spikes = [
        extract_biological_spikes(
            window_voltage[:, repetition],
            dt_ms=DT_MS,
            threshold_mv=BIOLOGICAL_THRESHOLD_MV,
        )
        for repetition in range(window_voltage.shape[1])
    ]

    fitting_windows.append({
        "start_s": start_s,
        "end_s": end_s,
        "current": window_current,
        "voltage": window_voltage,
        "biological_spikes": window_spikes,
        "duration_ms": len(window_current) * DT_MS,
    })

fitting_window_summary = pd.DataFrame([
    {
        "start_s": window["start_s"],
        "end_s": window["end_s"],
        "duration_s": window["duration_ms"] / 1000.0,
        "mean_biological_spikes": np.mean([
            len(spikes)
            for spikes in window["biological_spikes"]
        ]),
    }
    for window in fitting_windows
])

display(fitting_window_summary.round(3))

## Cell 12 — Define the fitting objective

For each candidate parameter vector:

- simulate every fitting window independently;
- compare the model spike train with every biological repetition;
- average all valid coincidence scores;
- minimise the negative mean score.

A small penalty is applied only when a candidate produces no spikes or an implausibly high firing rate. Negative coincidence scores are retained.

In [ ]:
MAX_REASONABLE_RATE_HZ = 300.0


def vector_to_parameters(spec, parameter_vector):
    fitted = dict(zip(spec["fit_names"], parameter_vector))
    return {**spec["base_params"], **fitted}


def candidate_fit_score(label, parameter_vector):
    spec = MODEL_SPECS[label]
    parameters = vector_to_parameters(spec, parameter_vector)

    all_scores = []

    for window in fitting_windows:
        try:
            model_spikes = simulate_nest_model(
                model_name=spec["nest_model"],
                current_values_pA=window["current"],
                parameters=parameters,
            )
        except Exception:
            return -1e6

        duration_s = window["duration_ms"] / 1000.0
        model_rate_hz = len(model_spikes) / duration_s

        if len(model_spikes) == 0:
            return -1e3

        if model_rate_hz > MAX_REASONABLE_RATE_HZ:
            return -1e3 - model_rate_hz

        scores = [
            calculate_coincidence_factor(
                model_spikes=model_spikes,
                data_spikes=biological_spikes,
                duration_ms=window["duration_ms"],
                delta_ms=DELTA_MS,
            )
            for biological_spikes in window["biological_spikes"]
        ]

        all_scores.extend(scores)

    if not np.any(np.isfinite(all_scores)):
        return -1e6

    return float(np.nanmean(all_scores))


def fitting_objective(parameter_vector, label):
    return -candidate_fit_score(label, parameter_vector)

## Cell 13 — Fit each model

The settings below are deliberately modest for a first complete run. Once the code is stable, increase `MAXITER` and `POPSIZE` and verify that the chosen parameters are stable across random seeds.

This is parameter optimisation, not neural-network training.

In [ ]:
MAXITER = 8
POPSIZE = 5
OPTIMISATION_WORKERS = 1

fitted_models = {}

for label, spec in MODEL_SPECS.items():
    print("\n" + "=" * 70)
    print(f"Fitting {label} ({spec['nest_model']})")
    print("=" * 70)

    start_time = time.perf_counter()

    result = differential_evolution(
        func=fitting_objective,
        bounds=spec["bounds"],
        args=(label,),
        seed=SEED,
        maxiter=MAXITER,
        popsize=POPSIZE,
        tol=0.01,
        polish=False,
        workers=OPTIMISATION_WORKERS,
        updating="immediate",
        disp=True,
    )

    elapsed_s = time.perf_counter() - start_time
    fitted_parameters = vector_to_parameters(spec, result.x)
    best_fit_gamma = -float(result.fun)

    fitted_models[label] = {
        "nest_model": spec["nest_model"],
        "parameters": fitted_parameters,
        "fit_gamma": best_fit_gamma,
        "optimiser_success": bool(result.success),
        "optimiser_message": str(result.message),
        "elapsed_s": elapsed_s,
    }

    print("\nBest fitted values")
    for name in spec["fit_names"]:
        print(f"  {name}: {fitted_parameters[name]:.6g}")

    print(f"Mean fitting Gamma: {best_fit_gamma:.4f}")
    print(f"Elapsed time: {elapsed_s / 60:.2f} minutes")

## Cell 14 — Review fitted parameters

This table should be inspected before validation. Parameters sitting exactly on a search boundary usually indicate that the bound or model convention needs further investigation.

In [ ]:
fit_rows = []

for label, result in fitted_models.items():
    row = {
        "model": label,
        "nest_model": result["nest_model"],
        "fit_gamma": result["fit_gamma"],
        "elapsed_min": result["elapsed_s"] / 60.0,
    }

    for parameter_name, value in result["parameters"].items():
        row[parameter_name] = value

    fit_rows.append(row)

fitted_parameter_table = pd.DataFrame(fit_rows)
display(fitted_parameter_table.round(5))

## Cell 15 — Evaluate fitted models on the untouched validation current

Every model is now simulated once on the full 14 s validation current.

The raw model–experiment score is averaged across all biological repetitions. The final score is:

\[
\Gamma_A =
\frac{\Gamma_{\mathrm{model-experiment}}}
{\Gamma_{\mathrm{experiment-experiment}}}
\]

No validation information is used during fitting.

In [ ]:
validation_duration_ms = len(I_validation) * DT_MS
validation_results = {}

for label, fitted in fitted_models.items():
    print("\n" + "-" * 70)
    print(f"Validating {label}")

    model_spikes = simulate_nest_model(
        model_name=fitted["nest_model"],
        current_values_pA=I_validation,
        parameters=fitted["parameters"],
        report_ignored=True,
    )

    per_repetition_gamma = np.asarray([
        calculate_coincidence_factor(
            model_spikes=model_spikes,
            data_spikes=biological_spikes,
            duration_ms=validation_duration_ms,
            delta_ms=DELTA_MS,
        )
        for biological_spikes in validation_reliability["spike_trains"]
    ])

    raw_gamma = float(np.nanmean(per_repetition_gamma))
    gamma_A = raw_gamma / R_VALIDATION
    firing_rate_hz = len(model_spikes) / (validation_duration_ms / 1000.0)

    validation_results[label] = {
        "model_spikes": model_spikes,
        "spike_count": len(model_spikes),
        "firing_rate_hz": firing_rate_hz,
        "per_repetition_gamma": per_repetition_gamma,
        "raw_gamma": raw_gamma,
        "gamma_A": gamma_A,
    }

    print(f"Spikes: {len(model_spikes)}")
    print(f"Firing rate: {firing_rate_hz:.2f} Hz")
    print(f"Raw model–experiment Gamma: {raw_gamma:.4f}")
    print(f"Normalised Gamma_A: {gamma_A:.4f}")

## Cell 16 — Final results table

The standard deviation here is across repeated biological trials of the same neuron. It is not equivalent to the across-neuron standard deviation shown in Kobayashi et al.'s Figure 3C.

In [ ]:
result_rows = []

for label, result in validation_results.items():
    result_rows.append({
        "model": label,
        "validation_spikes": result["spike_count"],
        "validation_rate_hz": result["firing_rate_hz"],
        "raw_gamma_mean": result["raw_gamma"],
        "raw_gamma_sd_across_repetitions": np.nanstd(
            result["per_repetition_gamma"],
            ddof=1,
        ),
        "biological_reliability": R_VALIDATION,
        "gamma_A": result["gamma_A"],
        "fit_gamma": fitted_models[label]["fit_gamma"],
    })

results_table = (
    pd.DataFrame(result_rows)
    .sort_values("gamma_A", ascending=False)
    .reset_index(drop=True)
)

display(results_table.round(4))

## Cell 17 — Plot the held-out predictive scores

The bars show \(\Gamma_A\). Error bars are derived from the variation in raw model–experiment coincidence across biological repetitions and scaled by the fixed validation reliability.

In [ ]:
plot_table = results_table.copy()

labels = plot_table["model"].to_list()
scores = plot_table["gamma_A"].to_numpy()
score_errors = (
    plot_table["raw_gamma_sd_across_repetitions"].to_numpy()
    / R_VALIDATION
)

plt.figure(figsize=(8, 5.5))
bars = plt.bar(
    labels,
    scores,
    yerr=score_errors,
    capsize=5,
    edgecolor="black",
    alpha=0.85,
)

for bar, score in zip(bars, scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{score:.3f}",
        ha="center",
        va="bottom",
    )

plt.axhline(1.0, linestyle="--", linewidth=1, label="Biological reliability ceiling")
plt.ylabel(r"Normalised predictive score, $\Gamma_A$")
plt.xlabel("Neuron model")
plt.title("Held-out spike-time prediction")
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## Cell 18 — Raster comparison on a short validation interval

This visual check is often more informative than the aggregate score. It shows whether a model is consistently early, late, overactive or underactive.

The validation spike times returned by NEST begin at 0 ms, so they align directly with the validation interval.

In [ ]:
RASTER_START_S = 2.0
RASTER_END_S = 5.0

raster_start_ms = RASTER_START_S * 1000.0
raster_end_ms = RASTER_END_S * 1000.0

plt.figure(figsize=(13, 7))

row = 0

# Plot a few biological repetitions.
for repetition, spikes in enumerate(
    validation_reliability["spike_trains"][:5]
):
    selected = spikes[
        (spikes >= raster_start_ms) &
        (spikes < raster_end_ms)
    ]

    plt.vlines(
        selected / 1000.0,
        row + 0.1,
        row + 0.9,
        linewidth=1,
    )
    row += 1

separator_row = row
row += 1

# Plot all model spike trains.
model_rows = {}

for label, result in validation_results.items():
    selected = result["model_spikes"][
        (result["model_spikes"] >= raster_start_ms) &
        (result["model_spikes"] < raster_end_ms)
    ]

    plt.vlines(
        selected / 1000.0,
        row + 0.1,
        row + 0.9,
        linewidth=1.2,
    )

    model_rows[label] = row
    row += 1

plt.axhline(separator_row, linewidth=1, linestyle="--")

yticks = list(range(5)) + list(model_rows.values())
yticklabels = [
    f"Biological {i + 1}" for i in range(5)
] + list(model_rows.keys())

plt.yticks(yticks, yticklabels)
plt.xlabel("Time within validation interval (s)")
plt.title("Biological and predicted spike times")
plt.xlim(RASTER_START_S, RASTER_END_S)
plt.tight_layout()
plt.show()

## Cell 19 — Optional blind-current predictions

This final cell applies the fitted models to the 39–60 s current. These spike trains can be exported for a challenge submission or later comparison, but no local \(\Gamma\) can be calculated because the biological voltage is withheld.

In [ ]:
blind_predictions = {}

for label, fitted in fitted_models.items():
    blind_spikes = simulate_nest_model(
        model_name=fitted["nest_model"],
        current_values_pA=I_blind,
        parameters=fitted["parameters"],
    )

    blind_predictions[label] = blind_spikes

    output_path = Path(
        f"blind_prediction_{label.lower()}.csv"
    )

    pd.DataFrame({
        "spike_time_ms": blind_spikes,
    }).to_csv(output_path, index=False)

    print(
        f"{label}: {len(blind_spikes)} spikes written to "
        f"{output_path.resolve()}"
    )